In [ ]:
import pandas as pd

# Files with Year 2025
files_2025 = [
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export.csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (1).csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (2).csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (3).csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (4).csv"
]

# Files with Year 2024
files_2024 = [
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (5).csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (6).csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (7).csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (8).csv",
    r"C:\Users\TrevorWhite\Downloads\Hitting Metrics Export (9).csv"
]

# Load and label files
def load_files(file_list, year):
    dfs = []
    for file in file_list:
        df = pd.read_csv(file)
        df['Year'] = year
        dfs.append(df)
    return dfs

dfs_2025 = load_files(files_2025, 2025)
dfs_2024 = load_files(files_2024, 2024)

# Combine and deduplicate
all_data = pd.concat(dfs_2025 + dfs_2024, ignore_index=True)
all_data = all_data.drop_duplicates()

# Preview
all_data.head()


In [ ]:
all_data['FULLNAME'] = all_data['playerFullName'].str.upper().str.replace(r'\s+', '', regex=True)
all_data.head()


In [ ]:
import pandas as pd
import numpy as np

# ——— your existing all_data DataFrame ———
# all_data = pd.read_csv(...) or however you’ve created it

# 1) Filter out rows where Swing# is 0
filtered = all_data[all_data['Swing#'] != 0].copy()

# 2) Coerce numeric columns to numeric dtype
numeric_cols = [
    'Take#', 'Swing#', 'SLG', 'wOBA', 'xWOBA', 'HH%', 'EV', 'EV90',
    'LA', 'GB/FB', 'Whiff%', 'Z-Con%', 'zSwing%', 'Chase%', '2kChase%'
]

# Before coercing to numeric, strip “%” and convert to a fraction
percent_cols = [col for col in filtered.columns if col.endswith('%')]
for col in percent_cols:
    filtered[col] = (
        pd.to_numeric(
            filtered[col].astype(str).str.rstrip('%'),
            errors='coerce'
        )
        / 100.0
    )



for col in numeric_cols:
    filtered[col] = pd.to_numeric(filtered[col], errors='coerce')

# 3) Check distinct player count before aggregation
player_count_before = filtered['playerId'].nunique()

# 4) Define "other" columns to carry forward first non-null
other_cols = [c for c in filtered.columns 
              if c not in numeric_cols + ['Year']]

# 5) Aggregation function, injecting playerId into the result
def aggregate_player(group):
    player_id = group['playerId'].iloc[0]
    row25 = group[group['Year'] == 2025]
    # If there's a 2025 row with Swing# > 50, return it intact
    if not row25.empty and row25['Swing#'].iloc[0] > 50:
        out = row25.iloc[0].to_dict()
        out['playerId'] = player_id
        return pd.Series(out)
    # Otherwise, build merged record
    out = {}
    # – average numeric columns
    for col in numeric_cols:
        out[col] = group[col].mean()
    # – first non-null for other columns
    for col in other_cols:
        nonnull = group[col].dropna()
        out[col] = nonnull.iloc[0] if len(nonnull) else np.nan
    # – set combined year and playerId
    out['Year']     = 2425
    out['playerId'] = player_id
    return pd.Series(out)

# 6) Manually iterate over groups to preserve all columns reliably
results = []
for pid, grp in filtered.groupby('playerId'):
    aggregated_row = aggregate_player(grp)
    results.append(aggregated_row)

aggregated = pd.DataFrame(results)

# 7) Validate same count of distinct players
player_count_after = aggregated['playerId'].nunique()
assert player_count_before == player_count_after, (
    f"Player count mismatch: before={player_count_before}, after={player_count_after}"
)
print(f"✔ Player count before & after aggregation: {player_count_before}")

# 8) Reorder columns so playerId & Year are first
cols = ['playerId', 'Year'] + [c for c in aggregated.columns if c not in ('playerId', 'Year')]
aggregated = aggregated[cols]

# Now `aggregated` has exactly one row per playerId, per your rules


In [ ]:
aggregated.head()

In [ ]:
aggregated.to_excel('C:/Users/TrevorWhite/Downloads/d1_hitting_stats_for_portal_04232025.xlsx', index=False)


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
